# BioPython Studio — Research Companion

**From a real Human Protein Atlas fluorescence image to an auditable analysis table.**

This notebook is original educational material linked to BioPython Studio. It assumes basic Python and uses real fluorescence **RGB JPEG composites**, not original quantitative 16-bit microscope data. Every output is an illustrative image-processing result: labels are *candidate connected regions*, **not validated cells, nuclei, puncta, aggregates, or exophers**.

**Sources and learning references:**
- [Human Protein Atlas](https://www.proteinatlas.org/) image `HPA-8346-22-C1-1`, [direct original JPEG](https://images.proteinatlas.org/8346/22_C1_1_blue_red_green.jpg), data attribution and [CC BY 4.0 license](https://www.proteinatlas.org/about/licence); Thul et al., *Science* 2017, DOI 10.1126/science.aal3321.
- [Image Processing for Beginners](https://guiwitz.github.io/PyImageCourse_beginner/README.html) — Guillaume Witz.
- [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/) — Jake VanderPlas.
- [Introduction to Bioimage Analysis](https://bioimagebook.github.io/) — Pete Bankhead.
- [Bio-image Analysis Notebooks](https://haesleinhuepf.github.io/BioImageAnalysisNotebooks/) — Robert Haase and contributors.
- [The Carpentries image processing course](https://datacarpentry.github.io/image-processing/).
- [Python for Biologists](https://www.pythonforbiologists.org/) — Martin Jones.

**Reproducibility rule:** Do not interpret a thresholded bright-pixel fraction or connected component as a biological object count without an independent annotated reference and documented acquisition parameters.

## 1. Prepare your Python environment

Run this notebook in Google Colab, JupyterLab or another Python 3 environment. Install only missing libraries. The training app uses Pyodide inside the browser; the notebook uses your notebook kernel and may need a separate installation.

In [ ]:
# In Colab/Jupyter run this only if the packages are missing:
# %pip install numpy pandas matplotlib scipy scikit-image pillow requests tifffile

import io, json, hashlib, sys
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from scipy import ndimage as ndi
from skimage import measure, morphology, filters

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__, 'Pandas:', pd.__version__)

## 2. Fetch the original HPA image and record provenance

We fetch the full JPEG from the HPA publisher, rather than redistributing the broken/truncated copies that caused an earlier app issue. This is one illustration of why provenance and file-integrity tests matter.

In [ ]:
SOURCE_URL = 'https://images.proteinatlas.org/8346/22_C1_1_blue_red_green.jpg'
response = requests.get(SOURCE_URL, timeout=35)
response.raise_for_status()
raw_bytes = response.content
assert len(raw_bytes) > 50_000, 'Image appears incomplete'
sha256 = hashlib.sha256(raw_bytes).hexdigest()
image_rgb = np.asarray(Image.open(io.BytesIO(raw_bytes)).convert('RGB'))
assert image_rgb.ndim == 3 and image_rgb.shape[2] == 3
print('Shape:', image_rgb.shape, 'dtype:', image_rgb.dtype)
print('Bytes:', len(raw_bytes), 'SHA256:', sha256[:20])
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image_rgb)
ax.set_title('Human Protein Atlas fluorescence composite')
ax.axis('off')
plt.show()

## 3. Select one intensity channel

The displayed blue channel is from a browser-decoded JPEG. It should **not** be treated as a calibrated fluorescence intensity image from the microscope. Explore how dtype and image shape affect calculations.

In [ ]:
image = image_rgb[:, :, 2]
assert image.dtype == np.uint8
print('Height, width:', image.shape)
print('Min, max, mean:', int(image.min()), int(image.max()), float(image.mean()))
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image, cmap='gray')
ax.set_title('HPA JPEG — blue channel (educational array)')
ax.axis('off')
plt.show()

## 4. Compare threshold choices and their effect

Thresholds depend on the image histogram. There is no universally correct threshold; a segmentation must be validated with appropriate reference annotations.

In [ ]:
cutoff = float(np.percentile(image, 85))
mask = image > cutoff
print('Threshold:', cutoff, 'bright pixels:', int(mask.sum()))
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(image, cmap='gray');axes[0].set_title('Blue channel')
axes[1].hist(image.ravel(), bins=32);axes[1].axvline(cutoff, color='red');axes[1].set_title('Intensity histogram')
axes[2].imshow(mask, cmap='gray');axes[2].set_title('Pixels selected')
for ax in axes: ax.tick_params(labelsize=7)
plt.tight_layout();plt.show()

## 5. Smooth and filter candidate regions

Gaussian smoothing and area filtering alter the segmentation. Such parameters must be documented because genuine puncta can disappear or adjacent objects can merge.

In [ ]:
smoothed = ndi.gaussian_filter(image.astype(float), sigma=1.2)
thresholded = smoothed > np.percentile(smoothed, 85)
cleaned = morphology.remove_small_objects(thresholded, min_size=25)
labels = measure.label(cleaned, connectivity=2)
print('Candidate regions:', int(labels.max()))
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(smoothed, cmap='gray');axes[0].set_title('Smoothed')
axes[1].imshow(cleaned, cmap='gray');axes[1].set_title('After small-region filter')
axes[2].imshow(labels, cmap='nipy_spectral');axes[2].set_title('Connected labels')
for ax in axes: ax.axis('off')
plt.tight_layout();plt.show()

## 6. Make a region-properties table

The output is one row per **segmented candidate**, not one row per known biological object. Region area is in pixels because calibrated pixel size is unavailable for this JPEG.

In [ ]:
properties = measure.regionprops_table(
    labels,
    intensity_image=image,
    properties=('label', 'area', 'mean_intensity', 'centroid')
)
regions = pd.DataFrame(properties)
assert {'label','area','mean_intensity'}.issubset(regions.columns)
print('Rows:', len(regions))
display(regions.head(10))

## 7. Connect measurements to experimentally meaningful metadata

These sample-level rows are **constructed teaching data**. They are not independent HPA biological experiments. The example demonstrates why a field identifier and biological-replicate identifier must remain distinct.

In [ ]:
fields = pd.DataFrame([
    {'sample_id':'CTRL-01','condition':'control','bio_rep':'C1','field':1,'signal':11},
    {'sample_id':'CTRL-01','condition':'control','bio_rep':'C1','field':2,'signal':13},
    {'sample_id':'CTRL-02','condition':'control','bio_rep':'C2','field':1,'signal':17},
    {'sample_id':'TRT-01','condition':'treated','bio_rep':'T1','field':1,'signal':19},
    {'sample_id':'TRT-01','condition':'treated','bio_rep':'T1','field':2,'signal':21},
    {'sample_id':'TRT-02','condition':'treated','bio_rep':'T2','field':1,'signal':16},
])
assert fields[['sample_id','field']].duplicated().sum() == 0
replicate_means = fields.groupby(['condition','bio_rep'])['signal'].mean().reset_index()
summary = replicate_means.groupby('condition')['signal'].agg(['count','mean','std'])
summary['sem'] = summary['std'] / np.sqrt(summary['count'])
display(replicate_means)
display(summary)

## 8. Show the actual biological-replicate observations

Plots should show individual biological replicates when practical. A mean + SEM alone cannot reveal an outlier, bimodality or small n.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for x, label in enumerate(summary.index):
    values = replicate_means.loc[replicate_means.condition == label, 'signal'].to_numpy()
    ax.scatter(np.full_like(values, x, dtype=float), values, alpha=.8, label=label)
    ax.errorbar(x, summary.loc[label, 'mean'], yerr=summary.loc[label, 'sem'],
                fmt='o', color='black', capsize=4)
ax.set_xticks(range(len(summary.index)), summary.index)
ax.set_ylabel('Illustrative signal per biological replicate')
ax.set_title('Biological-replicate-aware summary (teaching data)')
plt.tight_layout();plt.show()

## 9. Build a reusable, checked image-analysis function

Define the algorithm once. Make all assumptions explicit in inputs, and return data together with provenance. **An external calibrated pixel size is required for real physical-unit measurements.** This exercise intentionally returns pixel units only.

In [ ]:
def analyze_field(image, source_id, threshold_percentile=85, minimum_region_size=25):
    assert image.ndim == 2, 'Expected a single 2D image plane'
    assert 0 < threshold_percentile < 100
    clean = ndi.gaussian_filter(image.astype(float), sigma=1.2)
    mask = clean > np.percentile(clean, threshold_percentile)
    mask = morphology.remove_small_objects(mask, min_size=minimum_region_size)
    labels = measure.label(mask, connectivity=2)
    table = pd.DataFrame(measure.regionprops_table(
        labels, intensity_image=image,
        properties=('label', 'area', 'mean_intensity')
    ))
    table['source_id'] = source_id
    return {'candidate_regions':int(labels.max()), 'bright_pixels':int(mask.sum()),
            'mask':mask, 'regions':table}

result = analyze_field(image, 'HPA-8346-22-C1-1')
assert result['candidate_regions'] == len(result['regions'])
assert result['bright_pixels'] >= result['candidate_regions']
print({k:v for k,v in result.items() if k not in ('mask','regions')})

## 10. Save the analysis with provenance

Every export should allow another scientist to identify source data, algorithms and assumptions. Store the hash, original URL, notebook version and parameters alongside the table.

In [ ]:
OUTPUT = Path('biopython_outputs')
OUTPUT.mkdir(exist_ok=True)
result['regions'].to_csv(OUTPUT/'candidate_regions.csv', index=False)
provenance = {
    'source_url':SOURCE_URL, 'source_sha256':sha256,
    'image_kind':'RGB fluorescence JPEG composite (not raw quantitative intensity)',
    'selected_channel':'decoded RGB blue channel',
    'threshold_percentile':85, 'gaussian_sigma':1.2,
    'minimum_region_size_px':25,
    'annotation_validation':'not performed',
    'biological_object_count':'not established'
}
(OUTPUT/'provenance.json').write_text(json.dumps(provenance,indent=2))
print('Saved:', list(OUTPUT.glob('*')))

## 11. Optional: work with a real TIFF/OME-TIFF file

The HPA JPEG above cannot recover original microscope bit depth, axes, pixel calibration or acquisition metadata. To work on a real local TIFF/OME-TIFF, supply one from your own experiment or a suitably licensed dataset. [OME-TIFF reference samples](https://codelibre.gitlab.io/ome/ome-model/doc/ome-tiff/data.html) document the format and published biological examples.

The following cell **does nothing unless you provide a file path**. It deliberately does not pretend that a JPEG converted to TIFF is a raw microscope acquisition.

In [ ]:
TIFF_PATH = None  # e.g. 'my_multichannel_image.ome.tiff'
if TIFF_PATH:
    import tifffile
    with tifffile.TiffFile(TIFF_PATH) as tif:
        first_series = tif.series[0]
        print('Axis order:', first_series.axes, 'Shape:', first_series.shape)
        print('Stored dtype:', first_series.dtype)
        print('Has OME metadata:', bool(tif.ome_metadata))
        data = first_series.asarray()
        print('Pixel range:', float(data.min()), float(data.max()))
else:
    print('Optional TIFF task skipped: supply a real TIFF/OME-TIFF file to enable it.')

## 12. Independent project checklist

Before using this approach for publication or an invention:
1. Replace the publisher's composite JPEG with raw image data and metadata.
2. Confirm channel identity, dimension labels, pixel calibration, bit depth, and any preprocessing.
3. Collect manually annotated references from an appropriate biological object class.
4. Evaluate missed objects, false positives, touching objects, outliers and sensitivity to thresholds.
5. Preserve independent biological replicates and acquisition batches.
6. Pin the analysis environment and check that all cells run from a clean kernel.
7. Save code, outputs, provenance, exclusion decisions and limitations.

**This notebook is a working learning project, not a validated cell-counting algorithm.**